# Lecture: Diffusion Models — The Forward Process

In the previous chapters we saw two ways of generating images:

- **VAEs** (C2) learn a structured latent space but produce **blurry** samples.
- **GANs** (C3) produce **sharp** samples but train through an unstable two-player
  game and can suffer from **mode collapse**.

**Diffusion models** (Sohl-Dickstein et al., 2015; Ho et al., 2020) take a third
route that is both stable to train *and* produces sharp, diverse samples. The
idea is disarmingly simple:

1. **Forward process** — gradually destroy an image by adding a tiny bit of
   Gaussian noise over many steps, until nothing but pure noise remains.
2. **Reverse process** — train a neural network to undo one noising step at a
   time. Generating a new image then means starting from pure noise and running
   the learned denoiser backwards.

This first notebook is about **building intuition for the forward process only**
— there is **no training here**. We will see how noise is added, why a *schedule*
matters, and the single most important equation of diffusion models: the
**closed-form** expression that lets us jump to any noise level in one step.

The reverse process — the actual generative model — follows in **C4-2**.

### The forward process, one step at a time

The forward process is a fixed **Markov chain** that adds Gaussian noise over
$T$ steps according to a **variance schedule** $\beta_1, \dots, \beta_T$:

$$q(x_t \mid x_{t-1}) = \mathcal{N}\!\left(x_t;\ \sqrt{1-\beta_t}\, x_{t-1},\ \beta_t I\right)$$

Each step slightly shrinks the previous image (factor $\sqrt{1-\beta_t}$) and adds
a little noise (variance $\beta_t$). The $\beta_t$ are small (here between
$10^{-4}$ and $0.02$) and grow linearly with $t$.

Let us load one Fashion-MNIST image and apply this rule step by step.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.datasets import FashionMNIST

torch.manual_seed(0)

# Normalise to [-1, 1] — the convention used throughout the diffusion notebooks.
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

dataset = FashionMNIST(root="./data", train=True, download=True, transform=transform)
x0, label = dataset[0]          # a single image, shape (1, 28, 28)
print("Image shape:", x0.shape, "| pixel range:", (x0.min().item(), x0.max().item()))

In [ ]:
# Linear variance schedule (identical to the one used in C4-2)
T          = 1000
beta_start = 1e-4
beta_end   = 0.02

betas = torch.linspace(beta_start, beta_end, T)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(betas.numpy())
ax.set_xlabel("Diffusion step $t$")
ax.set_ylabel(r"$\beta_t$")
ax.set_title("Linear variance schedule")
plt.tight_layout()
plt.show()

### Iterating the chain

We now apply $q(x_t \mid x_{t-1})$ repeatedly, starting from the clean image
$x_0$. After enough steps the image becomes indistinguishable from a sample of
$\mathcal{N}(0, I)$ — all structure is gone.

In [ ]:
@torch.no_grad()
def forward_step(x_prev, t):
    """Apply one forward diffusion step q(x_t | x_{t-1})."""
    beta_t = betas[t]
    mean   = torch.sqrt(1.0 - beta_t) * x_prev
    return mean + torch.sqrt(beta_t) * torch.randn_like(x_prev)

# Iterate the Markov chain and store a few snapshots
snapshot_steps = [0, 50, 100, 200, 400, 700, 999]
x = x0.clone()
snapshots = {0: x.clone()}

for t in range(1, T):
    x = forward_step(x, t)
    if t in snapshot_steps:
        snapshots[t] = x.clone()

fig, axes = plt.subplots(1, len(snapshot_steps), figsize=(15, 2.5))
for ax, t in zip(axes, snapshot_steps):
    img = (snapshots[t].squeeze() + 1) / 2          # [-1,1] -> [0,1] for display
    ax.imshow(img.clamp(0, 1), cmap="gray")
    ax.set_title(f"t = {t}", fontsize=9)
    ax.axis("off")
plt.suptitle("Iterative forward process: clean image $\\to$ pure noise", y=1.08)
plt.tight_layout()
plt.show()

### The key trick: jumping to any step in closed form

Iterating one step at a time is wasteful — during training we need $x_t$ for a
*random* $t$ thousands of times. Fortunately, because each step is a linear
Gaussian map, the composition has a **closed form**. Defining

$$\alpha_t = 1 - \beta_t, \qquad \bar\alpha_t = \prod_{s=1}^{t} \alpha_s,$$

one can sample $x_t$ directly from the clean image $x_0$ in a **single step**:

$$\boxed{\,x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1 - \bar\alpha_t}\,\varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I)\,}$$

This is *the* equation that makes diffusion training practical. It says: a noisy
image at level $t$ is just a **weighted blend** of the original image and pure
noise, with weights set entirely by $\bar\alpha_t$. Let us verify it visually
against the iterative chain above.

In [ ]:
alphas     = 1.0 - betas
alphas_bar = torch.cumprod(alphas, dim=0)

@torch.no_grad()
def q_sample(x0, t, noise=None):
    """Closed-form sample x_t ~ q(x_t | x_0)."""
    if noise is None:
        noise = torch.randn_like(x0)
    sa = torch.sqrt(alphas_bar[t])
    sb = torch.sqrt(1.0 - alphas_bar[t])
    return sa * x0 + sb * noise

# Same snapshot steps, but now sampled directly from x0 (no iteration)
fig, axes = plt.subplots(1, len(snapshot_steps), figsize=(15, 2.5))
for ax, t in zip(axes, snapshot_steps):
    xt  = q_sample(x0, t)
    img = (xt.squeeze() + 1) / 2
    ax.imshow(img.clamp(0, 1), cmap="gray")
    ax.set_title(f"t = {t}", fontsize=9)
    ax.axis("off")
plt.suptitle("Closed-form forward process $x_t = \\sqrt{\\bar\\alpha_t}\\,x_0 + \\sqrt{1-\\bar\\alpha_t}\\,\\varepsilon$", y=1.08)
plt.tight_layout()
plt.show()

### How fast does the signal vanish?

The quantity $\bar\alpha_t$ is the **fraction of the original signal** that
survives at step $t$, and $1 - \bar\alpha_t$ is the **fraction of noise**. Plotting
both makes the schedule's effect concrete: the signal decays smoothly to zero,
so by $t = T$ the image is essentially pure noise — exactly the prior
$\mathcal{N}(0, I)$ from which the reverse process will start in C4-2.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(alphas_bar.numpy(),       label=r"signal weight $\sqrt{\bar\alpha_t}^2 = \bar\alpha_t$")
ax.plot((1 - alphas_bar).numpy(), label=r"noise weight $1 - \bar\alpha_t$")
ax.set_xlabel("Diffusion step $t$")
ax.set_ylabel("Fraction")
ax.set_title("Signal vs. noise over the forward process")
ax.legend()
plt.tight_layout()
plt.show()

print(f"alpha_bar at t=0:   {alphas_bar[0]:.4f}  (almost all signal)")
print(f"alpha_bar at t=999: {alphas_bar[-1]:.6f}  (almost no signal left)")

### A whole batch at one noise level

Finally, to connect with training: in C4-2 the network is shown many images, each
noised to a **random** timestep, and must predict the noise $\varepsilon$ that was
added. Here is a batch noised to the *same* level $t$ so you can see how much
structure remains at different points along the schedule.

In [ ]:
xs = torch.stack([dataset[i][0] for i in range(8)])   # (8, 1, 28, 28)

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for row, t in enumerate([0, 100, 300, 600]):
    noise = torch.randn_like(xs)
    xt = q_sample(xs, t, noise)
    for col in range(8):
        img = (xt[col].squeeze() + 1) / 2
        axes[row, col].imshow(img.clamp(0, 1), cmap="gray")
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(f"t = {t}", rotation=0, labelpad=30, fontsize=11, va="center")
plt.suptitle("A batch of Fashion-MNIST images at increasing noise levels", y=0.99)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

---
## Try It Yourself — Explore the Forward Process

Work in pairs. **Predict first, then run, then explain in one sentence.**

**A. Iterative vs. closed form.** The two image strips (iterative chain and
`q_sample`) use *different* random noise but should look statistically
equivalent at each $t$. Why are they allowed to differ pixel-by-pixel yet agree
in "how noisy" they look? (Hint: both are samples from the *same* distribution
$q(x_t \mid x_0)$.)

**B. Change the schedule.** Set `beta_end = 0.005` and re-run. Does the image
reach pure noise by $t = 999$? Look at the $\bar\alpha_t$ curve — what value does
it end at now, and why does that matter for the reverse process in C4-2?

**C. More vs. fewer steps.** Keep `beta_start`/`beta_end` fixed but set
`T = 200`. What happens to the *per-step* amount of noise, and to the smoothness
of the transition? What trade-off does a small $T$ create between training cost
and sample quality?

**D. Where is the information?** At which timestep does the *class* of the
clothing item become unrecognisable to you? This is roughly the point beyond
which the network can no longer recover the original — and it motivates why most
of the schedule's "interesting" learning happens at **intermediate** noise
levels.